# Scientific calculator: plan a dilution

**Facilitator reference solution**

## Before you start
**Time:** 45–75 minutes. **Experience:** no previous Python required.

In Google Colab, open **File > Upload notebook** and select this `.ipynb` file. Save a copy to your Drive if needed. Run cells from top to bottom using the triangle beside each code cell or **Shift + Enter**. Text cells explain the next step. A variable is a name for a value; a function is a reusable set of instructions.

All example data are synthetic and generated inside this notebook. No uploads, paid tools, API keys or extra datasets are required. Imports use NumPy, pandas, Matplotlib and, for signal analysis, SciPy. If using a local Jupyter installation, install the packages listed in `requirements.txt` first.

**Your goal:** run the working example, make the guided changes, check the result, then personalise it for your subject. `TODO` comments identify changes to try. The starter already runs successfully before you edit it.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.grid': True})

## 1. The scientific model
For a simple dilution, **C₁V₁ = C₂V₂**. Therefore the stock volume is **V₁ = C₂V₂/C₁**.

Use the same concentration units for both concentrations and mL for both volumes. This educational model assumes a simple dilution with additive volumes. Check the method before applying it to laboratory work.

We will turn the equation into a function. A function accepts inputs and returns results, making a calculation reusable.

In [2]:
def dilution_plan(stock_conc, target_conc, final_volume_ml):
    values = np.asarray([stock_conc, target_conc, final_volume_ml], dtype=float)
    if not np.isfinite(values).all():
        raise ValueError('Use finite numeric inputs.')
    if stock_conc <= 0 or final_volume_ml <= 0 or target_conc < 0:
        raise ValueError('Stock and final volume must be positive; target cannot be negative.')
    if target_conc > stock_conc:
        raise ValueError('Dilution cannot make a target more concentrated than the stock.')
    stock_volume_ml = target_conc * final_volume_ml / stock_conc
    solvent_volume_ml = final_volume_ml - stock_volume_ml
    return stock_volume_ml, solvent_volume_ml

## 2. Choose your inputs
The inputs below use mmol/L for concentration. A `#` starts a comment that Python ignores. Change the values and run this cell again.

In [3]:
# TODO: change these inputs while keeping consistent units.
stock_conc = 100.0        # mmol/L
target_conc = 10.0        # mmol/L
final_volume_ml = 50.0    # mL
stock_ml, solvent_ml = dilution_plan(stock_conc, target_conc, final_volume_ml)
print(f'Stock volume: {stock_ml:.2f} mL')
print(f'Solvent volume: {solvent_ml:.2f} mL')

Stock volume: 5.00 mL
Solvent volume: 45.00 mL


## 3. Check a known answer
For a 100 mmol/L stock and a 10 mmol/L target with a final volume of 50 mL, the answer is 5 mL stock and 45 mL solvent. `assert` raises an error if a check fails. `isclose` allows tiny rounding differences.

In [4]:
test_stock, test_solvent = dilution_plan(100, 10, 50)
assert np.isclose(test_stock, 5)
assert np.isclose(test_solvent, 45)
assert np.isclose(test_stock + test_solvent, 50)
assert dilution_plan(100, 100, 50) == (50, 0)
assert dilution_plan(100, 0, 50) == (0, 50)
try:
    dilution_plan(10, 100, 50)
except ValueError as error:
    print('Expected invalid-input message:', error)
else:
    raise AssertionError('An impossible dilution was accepted.')
print('Known-answer and boundary checks passed.')

Expected invalid-input message: Dilution cannot make a target more concentrated than the stock.
Known-answer and boundary checks passed.


## 4. Make a dilution table
A loop repeats a calculation for different target concentrations. The resulting table is saved as a CSV.

In [5]:
targets = [2, 5, 10, 20]  # mmol/L
rows = []
for target in targets:
    stock, solvent = dilution_plan(100, target, 50)
    rows.append({'target_mmol_L': target, 'stock_ml': stock, 'solvent_ml': solvent})
plan = pd.DataFrame(rows)
print(plan.to_string(index=False))
plan.to_csv('dilution_plan.csv', index=False)

 target_mmol_L  stock_ml  solvent_ml
             2       1.0        49.0
             5       2.5        47.5
            10       5.0        45.0
            20      10.0        40.0


## 5. Your tasks
- **Beginner:** calculate a 25 mL solution at 5 mmol/L from a 50 mmol/L stock. Check by hand.
- **Core:** add two more target concentrations to the table.
- **Personalise:** build an Ohm’s law calculator (`V = I × R`), a density calculator (`density = mass / volume`) or a percentage-yield calculator.
- **Stretch:** write a second function with input checks and a known-answer test.

**Hint for density:** reject zero or negative volume and state whether mass is in g and volume in mL.

In [6]:
# TODO: add your own calculation or function below.
my_stock_ml, my_solvent_ml = dilution_plan(50, 5, 25)
print(f'My example: {my_stock_ml:.2f} mL stock and {my_solvent_ml:.2f} mL solvent')

My example: 2.50 mL stock and 22.50 mL solvent


## Worked extension
Use this after the guided activities. Other correct approaches are welcome.

In [7]:
assert np.allclose(dilution_plan(50, 5, 25), (2.5, 22.5))
def density_g_ml(mass_g, volume_ml):
    if not np.isfinite([mass_g, volume_ml]).all() or mass_g < 0 or volume_ml <= 0:
        raise ValueError('Mass must be finite and non-negative; volume finite and positive.')
    return mass_g / volume_ml
assert np.isclose(density_g_ml(10, 5), 2)
print(f'Density of 10 g in 5 mL: {density_g_ml(10, 5):.2f} g/mL')

Density of 10 g in 5 mL: 2.00 g/mL


## Save and explain your project
1. Give the notebook your own project title.
2. Add a text cell explaining the question, inputs, output and one limitation.
3. Restart the session/kernel and run every cell in order. Does it still work?
4. Download the notebook as `.ipynb` and save your output files from the file sidebar. In Colab, runtime files can disappear when the session ends.

**Reflection:** What did you change? How did you check it? What would you add next?